# Réplica FA-CNN-RNN · IoTID20

Réplica de **Zayed, Alshathri & El-Shafai (2026)**, *Firefly Algorithm Optimized Hybrid Deep Learning Framework for Intrusion Detection in IoT Environments*, IJCIS 19:158. [DOI](https://doi.org/10.1007/s44196-026-01178-2)

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU (T4)**.

Tiempo aproximado con GPU: 35–45 min en modo fiel (batch 64) y 10–15 min con `FAST = True` (batch 256).

In [ ]:
REPO_URL = "https://github.com/Brunoxdc/fa-cnn-rnn-iotid20.git"
FAST = False   # True = batch 256 (más rápido, menos fiel al paper)

In [ ]:
import os
if not os.path.exists("fa-cnn-rnn-iotid20"):
    !git clone -q {REPO_URL}
%cd fa-cnn-rnn-iotid20
!pip install -q imbalanced-learn kagglehub

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Descargar IoTID20 (Kaggle, público, sin login)

In [ ]:
import kagglehub
DATA_DIR = kagglehub.dataset_download("rohulaminlabid/iotid20-dataset")
!ls -lh "{DATA_DIR}"

## 2. Ejecutar los experimentos E0–E4
E0 auditoría · E1 réplica fiel · E2 sin Firefly · E3 Random Forest · E4 sin duplicados

In [ ]:
flag = "--fast" if FAST else ""
!python -u run_experiments.py --data "{DATA_DIR}" {flag} 2>&1 | grep -v -E "I0000|W0000|E0000|oneDNN|cpu_feature|GrapplerHook|empty op name|does not specify"

## 3. Resultados

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
s = json.load(open("results/summary.json"))
print("Auditoría:", s["auditoria"])
print("Firefly eligió", s["firefly"]["n_seleccionadas"], "de", s["firefly"]["n_total"], "columnas:")
print(s["firefly"]["features_seleccionadas"])
pd.read_csv("results/metrics.csv").round(4)

In [ ]:
for f in ["comparacion_paper_vs_replica", "cm_fa_cnn_rnn", "cm_fa_cnn_rnn_dedup", "cm_rf",
          "firefly_convergencia", "entrenamiento_fa_cnn_rnn"]:
    print(f); display(Image(f"figures/{f}.png", width=520))

## 4. Guardar resultados
Actualiza el README con la tabla y descarga un zip con `results/`, `figures/` y el README.

In [ ]:
!python scripts/update_readme.py
!zip -qr resultados_replica.zip results figures README.md
from google.colab import files
files.download("resultados_replica.zip")

Para subir los resultados al repo, descomprime el zip en tu copia local y ejecuta:
```
git add results figures README.md
git commit -m "Resultados de la réplica en IoTID20"
git push
```